In [ ]:
import tangram as tg

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import scanpy as sc
import squidpy as sq
from anndata import AnnData

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")
print(f"tangram=={tg.__version__}")

squidpy==1.6.5
tangram==1.0.4


In [ ]:
# load ST and scRNA-seq data
adata_st = sc.read_h5ad('./data/simulated_data.h5ad')
adata_sc = sc.read_h5ad('./data/sc_reference.h5ad')

In [ ]:
adata_st.raw = adata_st.copy()
sc.pp.normalize_total(adata_st, target_sum=1e4)
sc.pp.log1p(adata_st)

adata_sc.raw = adata_sc.copy()
sc.pp.normalize_total(adata_sc, target_sum=1e4)
sc.pp.log1p(adata_sc)

In [ ]:
# Identify marker genes
sc.tl.rank_genes_groups(adata_sc, groupby="label", use_raw=False)
markers_df = pd.DataFrame(adata_sc.uns["rank_genes_groups"]["names"]).iloc[0:100, :]
markers = list(np.unique(markers_df.melt().value.values))
len(markers)

In [5]:
tg.pp_adatas(adata_sc, adata_st, genes=markers)

##### Find Alignment

In [8]:
ad_map = tg.map_cells_to_space(adata_sc, adata_st,
    mode="cells",
#     mode="clusters",
#     cluster_label='cell_subclass',  # .obs field w cell types
    density_prior='rna_count_based',
    num_epochs=500,
    device="cuda:2",
    # device='cpu',
)

INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 99 genes and rna_count_based density_prior in cells mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.533, KL reg: 0.255
Score: 0.956, KL reg: 0.007
Score: 0.970, KL reg: 0.003
Score: 0.975, KL reg: 0.002
Score: 0.977, KL reg: 0.001


INFO:root:Saving results..


##### Cell type maps

In [9]:
tg.project_cell_annotations(ad_map, adata_st, annotation="label")
annotation_list = list(pd.unique(adata_sc.obs['label']))

INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


In [ ]:
tg.plot_cell_annotation_sc(adata_st, annotation_list, spot_size=1, scale_factor=1, perc=0.02)

In [10]:
ad_ge = tg.project_genes(adata_map=ad_map, adata_sc=adata_sc)
ad_ge

AnnData object with n_obs × n_vars = 10594 × 28173
    obs: 'x', 'y', 'x_pixel', 'y_pixel', 'B_Cells', 'CD4+_T_Cells', 'CD8+_T_Cells', 'DCIS_1', 'DCIS_2', 'Endothelial', 'IRF7+_DCs', 'Invasive_Tumor', 'LAMP3+_DCs', 'Macrophages_1', 'Macrophages_2', 'Mast_Cells', 'Myoepi_ACTA2+', 'Myoepi_KRT15+', 'Perivascular-Like', 'Prolif_Invasive_Tumor', 'Stromal', 'cell_count', 'uniform_density', 'rna_count_based_density'
    var: 'mt', 'n_cells', 'sparsity', 'is_training'
    uns: 'training_genes', 'overlap_genes'

##### Image segmentation

In [6]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import numpy as np
from cellpose import models

In [ ]:
img = Image.open('./data/he.png')
img_arr = np.array(img)
img = sq.im.ImageContainer(img_arr, layer="image")

def cellpose_he(img, min_size=15, flow_threshold=0.4, cellprob_threshold=0.0, gpu=True):
    model = models.CellposeModel(gpu=gpu, pretrained_model="cpsam")
    masks, flows, styles = model.eval(
        img,
        min_size=min_size,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
    )
    return masks

In [ ]:
import os

img_path = "./result/tangram/segmented_image.zarr"
if os.path.exists(img_path):
    img = sq.im.ImageContainer.load(img_path)
else:
    sq.im.segment(
        img=img,
        layer="image",
        method=cellpose_he,
        channel=None, 
        flow_threshold=0.8,
        min_size=15,
        cellprob_threshold=0.0,
        layer_added = 'segmented_Cellpose'
    )
    img.save(img_path)

In [ ]:
from skimage.measure import regionprops
import numpy as np

seg_layer = img["segmented_Cellpose"]

seg_mask = seg_layer.values if hasattr(seg_layer, "values") else np.asarray(seg_layer)
seg_mask = np.squeeze(seg_mask) 
print("seg mask shape:", seg_mask.shape, "dtype:", seg_mask.dtype)

uniq = np.unique(seg_mask)
n_inst = int((uniq != 0).sum())
n_inst_fast = int(seg_mask.max())

print(f"Segmented Cell number: {n_inst}")
props = regionprops(seg_mask.astype(int))

centroids = np.array([prop.centroid for prop in props])

print("Number of centroids:", len(centroids))
print("First few centroids:", centroids[:5])

# 如果需要转换为 (x, y) 格式：
centroids_xy = centroids[:, [1, 0]]
print("First few centroids (x, y):", centroids_xy[:5])

seg mask shape: (17098, 51187) dtype: uint32
Segmented Cell number: 136957


Number of centroids: 136957
First few centroids: [[ 1722.33333333 46594.80555556]
 [ 1753.88345324 25736.33669065]
 [ 1779.47945205 46571.39726027]
 [ 1824.91666667 46645.84722222]
 [ 1831.11607143 46582.125     ]]
First few centroids (x, y): [[46594.80555556  1722.33333333]
 [25736.33669065  1753.88345324]
 [46571.39726027  1779.47945205]
 [46645.84722222  1824.91666667]
 [46582.125       1831.11607143]]


In [ ]:
# Visualize segmentation
crop = img.crop_corner(5000, 5000, size=1000)

fig, axes = plt.subplots(1, 2, figsize=(10, 20))
crop.show("image", channel=None, ax=axes[0])
_ = axes[0].set_title("H&E")
crop.show("segmented_Cellpose", cmap="jet", interpolation="none", ax=axes[1])
_ = axes[1].set_title("Cellpose segmentation")

In [9]:
h, w = img.shape[:2]

out = (adata_st.obs["x_pixel"] >= w) | (adata_st.obs["y_pixel"] >= h) \
    | (adata_st.obs["x_pixel"] < 0)  | (adata_st.obs["y_pixel"] < 0)

adata_st_filter = adata_st[~out].copy()

In [10]:
# define image layer to use for segmentation
features_kwargs = {
    "segmentation": {
        "label_layer": "segmented_Cellpose",
        "props": ["label", "centroid"],
        "channels": [1, 2],
    }
}

# calculate segmentation features
sq.im.calculate_image_features(
    adata_st_filter,
    img,
    layer="image",
    key_added="image_features",
    features_kwargs=features_kwargs,
    features="segmentation",
    mask_circle=False,
)

100%|██████████| 496/496 [00:09<00:00, 53.91/s]


In [9]:
# 自定义函数分配细胞到spot上

def cell2spot(adata, nucleus_center, spot_radius = None, spot_shape = "circle"):
    """
    Process a single spot to assign cells within its area.
    Args:
        spot_center (np.ndarray): Center coordinates of the spot.
        nucleus_center (np.ndarray): Center coordinates of the nuclei.
        spot_radius (int): Radius of the spot for circular shape.
        spot_shape (str): Shape of the spot, either "circle" or "square".
    """
    import numpy as np
    import pandas as pd
    
    spot_center = adata.obsm['spatial']
    if spot_radius is None:
        library_id = next(iter(adata.uns["spatial"]))
        spot_radius = adata.uns['spatial'][library_id]['scalefactors']['spot_diameter_fullres'] / 2
    
    n_spots = len(spot_center)
    n_cells = len(nucleus_center)
    
    # Calculate mask (n_spots, n_cells) - 这部分已经是向量化的
    if spot_shape == "circle":
        distances = np.sum((spot_center[:, :, np.newaxis] - nucleus_center.T) ** 2, axis=1)
        mask = distances < spot_radius ** 2
    elif spot_shape == "square":
        dx = np.abs(spot_center[:, 0:1] - nucleus_center[:, 0])
        dy = np.abs(spot_center[:, 1:2] - nucleus_center[:, 1])
        mask = (dx <= spot_radius) & (dy <= spot_radius)
    else:
        raise ValueError("spot_shape must be either 'circle' or 'square'")
    
    spot_cell_data = []
    for i in range(n_spots):
        spot_mask = mask[i]
        cell_ids = np.where(spot_mask)[0].tolist()
        centroids_raw = nucleus_center[spot_mask][:, [1, 0]]  # 交换 y,x 为 x,y
        cell_num = spot_mask.sum()
        
        # 转换 centroids 为标准格式：list of pairs
        if len(centroids_raw) == 0:
            centroids_formatted = [[np.nan, np.nan]]
        else:
            centroids_formatted = centroids_raw.tolist()
        
        spot_cell_data.append({
            "cell_ids": cell_ids,
            "centroids": centroids_formatted,
            "cell_num": cell_num
        })
    
    spot_cell_df = pd.DataFrame(spot_cell_data, index=adata.obs_names)   
    return spot_cell_df

spot_cell_df = cell2spot(adata_st, centroids_xy, spot_shape = "square")
adata_st.obsm['image_features'] = spot_cell_df
adata_st.obsm['image_features'].columns = ['cell_ids', 'segmentation_centroid', 'segmentation_label']

In [10]:
print(f"分割到的细胞数: {len(uniq)-1}")
print(f"自定义分配结果:{spot_cell_df['segmentation_label'].sum()}")
# print(f"squidpy分配结果:{squidpy_seg['segmentation_label'].sum()}")

分割到的细胞数: 117504
自定义分配结果:117455


##### Deconvolution and mapping

In [12]:
adata_st_filter.obs['cell_count'] = adata_st_filter.obsm['image_features']['segmentation_label']

In [13]:
ad_map = tg.map_cells_to_space(
    adata_sc,
    adata_st_filter,
    mode="constrained",
    target_count=adata_st_filter.obs.cell_count.sum(),
    density_prior=np.array(adata_st_filter.obs.cell_count) / adata_st_filter.obs.cell_count.sum(),
    num_epochs=2000,
    device="cuda:3",
    lambda_count = 1,
    # device='cpu',
)

INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 119 genes and customized density_prior in constrained mode...


Score: 0.849, KL reg: 0.181, Count reg: 166368.969, Lambda f reg: 84950.914
Score: 0.968, KL reg: 0.010, Count reg: 477.668, Lambda f reg: 17687.289
Score: 0.974, KL reg: 0.008, Count reg: 75.562, Lambda f reg: 7615.665
Score: 0.975, KL reg: 0.008, Count reg: 28.582, Lambda f reg: 5440.506
Score: 0.976, KL reg: 0.008, Count reg: 37.828, Lambda f reg: 4428.462
Score: 0.976, KL reg: 0.008, Count reg: 29.938, Lambda f reg: 3792.006
Score: 0.976, KL reg: 0.008, Count reg: 36.023, Lambda f reg: 3354.134
Score: 0.977, KL reg: 0.007, Count reg: 10.590, Lambda f reg: 3051.358
Score: 0.977, KL reg: 0.007, Count reg: 3.230, Lambda f reg: 2764.786
Score: 0.977, KL reg: 0.007, Count reg: 39.176, Lambda f reg: 2504.306
Score: 0.977, KL reg: 0.007, Count reg: 15.273, Lambda f reg: 2334.966
Score: 0.977, KL reg: 0.007, Count reg: 48.867, Lambda f reg: 2137.731
Score: 0.977, KL reg: 0.007, Count reg: 48.625, Lambda f reg: 1996.246
Score: 0.977, KL reg: 0.007, Count reg: 48.754, Lambda f reg: 1873.017


INFO:root:Saving results..


In [15]:
tg.project_cell_annotations(ad_map, adata_st_filter, annotation="label")
annotation_list = list(pd.unique(adata_sc.obs['label']))
# tg.plot_cell_annotation_sc(adata_st_filter, annotation_list, perc=0.02)

INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


In [ ]:
deconv_res = adata_st_filter.obsm['tangram_ct_pred']
deconv_res.sum(axis=1)
os.makedirs('./result/tangram/', exist_ok =True)
deconv_res.to_csv('./result/tangram/deconv.csv')

In [ ]:
tg.create_segment_cell_df(adata_st_filter)
print(adata_st_filter.uns["tangram_cell_segmentation"].head())

INFO:root:cell segmentation dataframe is saved in `uns` `tangram_cell_segmentation` of the spatial AnnData.
INFO:root:spot centroids is saved in `obsm` `tangram_spot_centroids` of the spatial AnnData.


10_100    [10_100_0, 10_100_1, 10_100_2, 10_100_3, 10_10...
10_106                       [10_106_0, 10_106_1, 10_106_2]
10_108    [10_108_0, 10_108_1, 10_108_2, 10_108_3, 10_10...
10_84     [10_84_0, 10_84_1, 10_84_2, 10_84_3, 10_84_4, ...
10_86     [10_86_0, 10_86_1, 10_86_2, 10_86_3, 10_86_4, ...
                                ...                        
8_90      [8_90_0, 8_90_1, 8_90_2, 8_90_3, 8_90_4, 8_90_...
8_92                                                     []
8_94                                       [8_94_0, 8_94_1]
8_96      [8_96_0, 8_96_1, 8_96_2, 8_96_3, 8_96_4, 8_96_...
8_98      [8_98_0, 8_98_1, 8_98_2, 8_98_3, 8_98_4, 8_98_...
Name: centroids_idx, Length: 496, dtype: object
  spot_idx             y            x centroids
0   10_100  47073.276834  4883.951613  10_100_0
1   10_100  47059.000000  5101.324600  10_100_1
2   10_100  47068.463383  5071.902921  10_100_2
3   10_100  47126.255233  5177.000000  10_100_3
4   10_100  47176.303286  4709.204358  10_100_4


In [ ]:
cent = adata_st_filter.obsm["tangram_spot_centroids"]
cent = np.asarray(cent, dtype=object)

if cent.ndim == 2 and cent.shape[1] == 1:
    cent = cent[:, 0]

adata_st_filter.obsm["tangram_spot_centroids"] = cent

tg.count_cell_annotations(
    ad_map,
    adata_sc,
    adata_st_filter,
    annotation="label",
)

adata_st_filter.obsm["tangram_ct_count"]['cell_n'].sum()

INFO:root:spatial cell count dataframe is saved in `obsm` `tangram_ct_count` of the spatial AnnData.


np.int64(38948)

In [ ]:
def df_to_cell_types(df, cell_types):
    """
    Utility function that "randomly" assigns cell coordinates in a voxel to known numbers of cell types in that voxel.
    Used for deconvolution.

    Args:
        df (DataFrame): Columns correspond to cell types.  Each row in the DataFrame corresponds to a voxel and
        specifies the known number of cells in that voxel for each cell type (int).
        The additional column 'centroids' specifies the coordinates of the cells in the voxel (sequence of (x,y) pairs).
        cell_types (sequence): Sequence of cell type names to be considered for deconvolution.
        Columns in 'df' not included in 'cell_types' are ignored for assignment.

    Returns:
        A dictionary <cell type name> -> <list of (x,y) coordinates for the cell type>
    """
    df_cum_sums = df[cell_types].cumsum(axis=1)

    df_c = df.copy()

    for i in df_cum_sums.columns:
        df_c[i] = df_cum_sums[i]

    cell_types_mapped = defaultdict(list)
    for i_index, i in enumerate(cell_types):
        for j_index, j in df_c.iterrows():
            start_ind = 0 if i_index == 0 else j[cell_types[i_index - 1]]
            end_ind = j[i]
            cell_types_mapped[i].extend(j["centroids"][start_ind:end_ind].tolist())
    return cell_types_mapped

def deconvolve_cell_annotations(adata_sp, filter_cell_annotation=None):
    """
    Assigns cell annotation to each segmented cell. Produces an AnnData structure that saves the assignment in its obs dataframe.

    Args:
        adata_sp (AnnData): Spatial AnnData structure.
        filter_cell_annotation (sequence): Optional. Sequence of cell annotation names to be considered for deconvolution. Default is None. When no values passed, all cell annotation names in adata_sp.obsm["tangram_ct_pred"] will be used.

    Returns:
        AnnData: Saves the cell annotation assignment result in its obs dataframe where each row representing a segmentation object, column 'x', 'y', 'centroids' contain its position and column 'cluster' is the assigned cell annotation.
    """

    if (
        "tangram_ct_count" not in adata_sp.obsm.keys()
        or "tangram_cell_segmentation" not in adata_sp.uns.keys()
    ):
        raise ValueError("Missing tangram parameters. Run `count_cell_annotations`.")

    segmentation_df = adata_sp.uns["tangram_cell_segmentation"]

    if filter_cell_annotation is None:
        filter_cell_annotation = pd.unique(
            list(adata_sp.obsm["tangram_ct_pred"].columns)
        )
    else:
        filter_cell_annotation = pd.unique(filter_cell_annotation)

    df_vox_cells = adata_sp.obsm["tangram_ct_count"]
    cell_types_mapped = df_to_cell_types(df_vox_cells, filter_cell_annotation)
    df_list = []
    for k in cell_types_mapped.keys():
        df = pd.DataFrame({"centroids": np.array(cell_types_mapped[k], dtype="object")})
        df["cluster"] = k
        df_list.append(df)
    cluster_df = pd.concat(df_list, axis=0)
    cluster_df.reset_index(inplace=True, drop=True)

    merged_df = segmentation_df.merge(cluster_df, on="centroids", how="inner")
    merged_df.drop(columns="spot_idx", inplace=True)
    merged_df.drop_duplicates(inplace=True)
    merged_df.dropna(inplace=True)
    merged_df.reset_index(inplace=True, drop=True)

    adata_segment = sc.AnnData(np.zeros(merged_df.shape), obs=merged_df)
    adata_segment.obsm["spatial"] = merged_df[["y", "x"]].to_numpy()
    adata_segment.uns = adata_sp.uns

    return adata_segment

adata_segment = deconvolve_cell_annotations(adata_st_filter)

In [ ]:
df = adata_segment.uns["tangram_cell_segmentation"].copy()

# 这些 NaN 行不是细胞，只是 explode 空列表产生的占位行，直接删掉
df = df.dropna(subset=["x", "y", "centroids"]).copy()

# 确保 centroids 是纯字符串列（避免 object 混合类型）
df["centroids"] = df["centroids"].astype(str)

adata_segment.uns["tangram_cell_segmentation"] = df
os.makedirs('./result/tangram/', exist_ok =True)
adata_segment.write_h5ad("./result/tangram/adata_st_seg.h5ad")